# Tabela descritiva: candidatos competitivos por magnitude × tipo de partido

Gera a tabela para inserção no Capítulo 3 (03-competicao-contida.qmd), após a definição de candidato competitivo e categorias de distrito.

**Candidato competitivo (ex-ante):** critérios a) e b) de @cheibubsin2020:
- a) incumbente: venceu ao menos uma eleição anterior (exceto Vereador)
- b) alcançou ≥10% do QE em alguma eleição anterior

**Tipo de partido (ex-ante):** bancada nacional prévia ≥20 cadeiras na data de início das convenções (bancada_partido_uf.csv), não os assentos ganhos na eleição em análise.

**Desagregação:** magnitude (Pequeno / Médio / Grande) × tipo de partido

In [ ]:
import os
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)


In [7]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tabulate import tabulate

sys.path.insert(0, str(Path("figs/src").resolve()))
from cap3_cs_features import gerar_features

df = pd.read_parquet("data/processed/rrd_df_novo.parquet")
rrd = df[df.ano_eleicao.isin([2018, 2022])].copy()
rrd = gerar_features(rrd)  # adiciona dm_cat, tipo_partido (ex-post), incumbente, etc.

print(f"Total de candidatos: {len(rrd):,}")
print(f"  2018: {(rrd.ano_eleicao == 2018).sum():,}")
print(f"  2022: {(rrd.ano_eleicao == 2022).sum():,}")

Total de candidatos: 17,305
  2018: 7,630
  2022: 9,675


In [8]:
# Substituir tipo_partido (ex-post em gerar_features) pela versão ex-ante
# baseada na bancada nacional prévia (data de início das convenções)

bancada = pd.read_csv("data/processed/bancada_partido_uf.csv")

# Harmonização mínima de nomes de partido no arquivo de bancada
harmonizacao = {"PP**": "PP", "PCdoB": "PC do B", "SD": "SOLIDARIEDADE"}
bancada["sg_partido"] = bancada["sg_partido"].replace(harmonizacao)

# Bancada nacional prévia: soma dos deputados estaduais por partido × ano
bancada_nacional = (
    bancada
    .groupby(["ano_eleicao", "sg_partido"])["n_deputados"]
    .sum()
    .reset_index()
    .rename(columns={"n_deputados": "bancada_nacional_previa"})
)

# Substituir tipo_partido
rrd = rrd.drop(columns=["tipo_partido", "cadeiras_nacionais"], errors="ignore")
rrd = rrd.merge(bancada_nacional, on=["ano_eleicao", "sg_partido"], how="left")
rrd["bancada_nacional_previa"] = rrd["bancada_nacional_previa"].fillna(0)
rrd["tipo_partido"] = np.where(
    rrd["bancada_nacional_previa"] >= 20, "Competitivo", "Menos competitivo"
)

# Verificação: PSL 2018 deve ser Menos competitivo (8 cadeiras prévias)
# PL 2022 deve ser Competitivo (77 cadeiras prévias)
print("Verificação PSL/PL — bancada prévia e classificação:")
check = (
    rrd[rrd.sg_partido.isin(["PSL", "PL"])]
    .groupby(["ano_eleicao", "sg_partido"])
    .agg(
        bancada_previa=("bancada_nacional_previa", "first"),
        tipo=("tipo_partido", "first"),
    )
    .reset_index()
)
print(check.to_string(index=False))

# Distribuição geral
print()
print("Partidos por tipo e ano:")
print(
    rrd.drop_duplicates(["ano_eleicao", "sg_partido"])
    .groupby(["ano_eleicao", "tipo_partido"])
    .size()
    .rename("n_partidos")
)

Verificação PSL/PL — bancada prévia e classificação:
 ano_eleicao sg_partido  bancada_previa              tipo
        2018        PSL             8.0 Menos competitivo
        2022         PL            77.0       Competitivo

Partidos por tipo e ano:
ano_eleicao  tipo_partido     
2018         Competitivo           9
             Menos competitivo    26
2022         Competitivo           9
             Menos competitivo    23
Name: n_partidos, dtype: int64


In [9]:
# Flag de candidato competitivo (ex-ante): incumbente OU histórico ≥10% QE
# Exclui 10pct_qe_eleicao_atual (ex-post)
rrd["competitivo_exante"] = (
    rrd["incumbente"].fillna(False)
    | rrd["alcancou_10pct_qe_hist"].fillna(False)
)

for ano in [2018, 2022]:
    sub = rrd[rrd.ano_eleicao == ano]
    n_comp = sub["competitivo_exante"].sum()
    n_tot = len(sub)
    print(f"{ano}: {n_comp:,} competitivos de {n_tot:,} ({n_comp/n_tot*100:.1f}%)")

2018: 886 competitivos de 7,630 (11.6%)
2022: 1,287 competitivos de 9,675 (13.3%)


In [10]:
# Tabela: magnitude × tipo_partido (ex-ante) × ano
tab = (
    rrd
    .groupby(["ano_eleicao", "dm_cat", "tipo_partido"], observed=True)
    .agg(
        n_total=("nr_candidato", "count"),
        n_comp=("competitivo_exante", "sum"),
    )
    .assign(pct=lambda x: x["n_comp"] / x["n_total"] * 100)
    .reset_index()
)

print(tab.to_string(index=False))

 ano_eleicao         dm_cat      tipo_partido  n_total  n_comp        pct
        2018 Pequeno (8–12)       Competitivo      476     146  30.672269
        2018 Pequeno (8–12) Menos competitivo     1234     117   9.481361
        2018  Médio (16–31)       Competitivo      616     173  28.084416
        2018  Médio (16–31) Menos competitivo     1503     122   8.117099
        2018 Grande (39–70)       Competitivo      940     197  20.957447
        2018 Grande (39–70) Menos competitivo     2861     131   4.578819
        2022 Pequeno (8–12)       Competitivo     1003     295  29.411765
        2022 Pequeno (8–12) Menos competitivo     1380      92   6.666667
        2022  Médio (16–31)       Competitivo     1310     288  21.984733
        2022  Médio (16–31) Menos competitivo     1825     141   7.726027
        2022 Grande (39–70)       Competitivo     1605     312  19.439252
        2022 Grande (39–70) Menos competitivo     2552     159   6.230408


In [11]:
# Linha de totais por ano
totais = (
    rrd
    .groupby("ano_eleicao")
    .agg(
        n_total=("nr_candidato", "count"),
        n_comp=("competitivo_exante", "sum"),
    )
    .assign(
        pct=lambda x: x["n_comp"] / x["n_total"] * 100,
        dm_cat="Total",
        tipo_partido="",
    )
    .reset_index()[["ano_eleicao", "dm_cat", "tipo_partido", "n_total", "n_comp", "pct"]]
)

tab_completa = pd.concat([tab, totais], ignore_index=True)

In [12]:
# Pivot final: linhas = (magnitude, tipo_partido), colunas = 2018 e 2022
def celula(row):
    return f"{int(row['n_comp']):,} / {int(row['n_total']):,} ({row['pct']:.1f}%)"

tab_completa["celula"] = tab_completa.apply(celula, axis=1)

pivot = (
    tab_completa
    .pivot_table(
        index=["dm_cat", "tipo_partido"],
        columns="ano_eleicao",
        values="celula",
        aggfunc="first",
    )
    .reset_index()
)

ordem_dm = ["Pequeno (8–12)", "Médio (16–31)", "Grande (39–70)", "Total"]
ordem_tp = ["Competitivo", "Menos competitivo", ""]
pivot["dm_cat"] = pd.Categorical(pivot["dm_cat"], categories=ordem_dm, ordered=True)
pivot["tipo_partido"] = pd.Categorical(pivot["tipo_partido"], categories=ordem_tp, ordered=True)
pivot = pivot.sort_values(["dm_cat", "tipo_partido"]).reset_index(drop=True)

pivot.columns.name = None
pivot = pivot.rename(columns={
    "dm_cat": "Magnitude",
    "tipo_partido": "Partido",
    2018: "2018",
    2022: "2022",
})

print(tabulate(pivot, headers="keys", tablefmt="pipe", showindex=False))

| Magnitude      | Partido           | 2018                | 2022                  |
|:---------------|:------------------|:--------------------|:----------------------|
| Pequeno (8–12) | Competitivo       | 146 / 476 (30.7%)   | 295 / 1,003 (29.4%)   |
| Pequeno (8–12) | Menos competitivo | 117 / 1,234 (9.5%)  | 92 / 1,380 (6.7%)     |
| Médio (16–31)  | Competitivo       | 173 / 616 (28.1%)   | 288 / 1,310 (22.0%)   |
| Médio (16–31)  | Menos competitivo | 122 / 1,503 (8.1%)  | 141 / 1,825 (7.7%)    |
| Grande (39–70) | Competitivo       | 197 / 940 (21.0%)   | 312 / 1,605 (19.4%)   |
| Grande (39–70) | Menos competitivo | 131 / 2,861 (4.6%)  | 159 / 2,552 (6.2%)    |
| Total          |                   | 886 / 7,630 (11.6%) | 1,287 / 9,675 (13.3%) |


In [13]:
# Competitivos × eleitos × magnitude
# Métricas:
#   taxa_comp             = % dos competitivos que se elegeram
#   taxa_ncomp            = % dos não-competitivos que se elegeram
#   pct_comp_entre_eleitos = % dos eleitos que eram ex-ante competitivos
#   razao_taxas           = taxa_comp / taxa_ncomp

def agregar_cross(g):
    return pd.Series({
        "comp_eleito":   int((g.competitivo_exante  & (g.eleito == 1)).sum()),
        "comp_neleito":  int((g.competitivo_exante  & (g.eleito == 0)).sum()),
        "ncomp_eleito":  int((~g.competitivo_exante & (g.eleito == 1)).sum()),
        "ncomp_neleito": int((~g.competitivo_exante & (g.eleito == 0)).sum()),
        "n_comp":        int(g.competitivo_exante.sum()),
        "n_ncomp":       int((~g.competitivo_exante).sum()),
        "n_eleitos":     int((g.eleito == 1).sum()),
        "n_total":       len(g),
    })

cross = (
    rrd
    .groupby(["ano_eleicao", "dm_cat"], observed=True)
    .apply(agregar_cross, include_groups=False)
    .reset_index()
)

totais_cross = (
    rrd
    .groupby("ano_eleicao")
    .apply(agregar_cross, include_groups=False)
    .reset_index()
    .assign(dm_cat="Total")
)

cross_full = pd.concat([cross, totais_cross], ignore_index=True)
cross_full["taxa_comp"]              = cross_full["comp_eleito"]  / cross_full["n_comp"]    * 100
cross_full["taxa_ncomp"]             = cross_full["ncomp_eleito"] / cross_full["n_ncomp"]   * 100
cross_full["pct_comp_entre_eleitos"] = cross_full["comp_eleito"]  / cross_full["n_eleitos"] * 100
cross_full["razao_taxas"]            = cross_full["taxa_comp"]    / cross_full["taxa_ncomp"]

# Exibição
print(f"{'─'*82}")
print(f"  Competitivos × eleitos × magnitude")
print(f"{'─'*82}")
print(f"  {'Magnitude':<16} {'Ano'}  {'Comp: eleitos/total (taxa)':>26}  {'N-comp: eleitos/total (taxa)':>28}  {'%eleitos=comp':>13}  {'Razão':>6}")
print(f"{'─'*82}")

ordem_dm = ["Pequeno (8–12)", "Médio (16–31)", "Grande (39–70)", "Total"]
for dm in ordem_dm:
    for ano in [2018, 2022]:
        r = cross_full[(cross_full["dm_cat"] == dm) & (cross_full["ano_eleicao"] == ano)]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        comp_str  = f"{r.comp_eleito:>3}/{r.n_comp:<5} ({r.taxa_comp:4.1f}%)"
        ncomp_str = f"{r.ncomp_eleito:>3}/{r.n_ncomp:<6} ({r.taxa_ncomp:4.1f}%)"
        print(f"  {dm:<16} {ano}  {comp_str:>26}  {ncomp_str:>28}  {r.pct_comp_entre_eleitos:>12.1f}%  {r.razao_taxas:>5.1f}×")
    if dm != "Total":
        print()

print(f"{'─'*82}")
print()
print("Comp: eleitos/total   = eleitos entre competitivos / total competitivos (taxa de eleição)")
print("%eleitos=comp         = % dos eleitos naquele segmento que eram ex-ante competitivos")
print("Razão                 = taxa eleição comp ÷ taxa eleição não-comp")

──────────────────────────────────────────────────────────────────────────────────
  Competitivos × eleitos × magnitude
──────────────────────────────────────────────────────────────────────────────────
  Magnitude        Ano  Comp: eleitos/total (taxa)  N-comp: eleitos/total (taxa)  %eleitos=comp   Razão
──────────────────────────────────────────────────────────────────────────────────
  Pequeno (8–12)   2018            86/263   (32.7%)             43/1447   ( 3.0%)          66.7%   11.0×
  Pequeno (8–12)   2022            83/387   (21.4%)             46/1996   ( 2.3%)          64.3%    9.3×

  Médio (16–31)    2018           127/295   (43.1%)             49/1824   ( 2.7%)          72.2%   16.0×
  Médio (16–31)    2022           145/429   (33.8%)             31/2706   ( 1.1%)          82.4%   29.5×

  Grande (39–70)   2018           125/328   (38.1%)             83/3473   ( 2.4%)          60.1%   15.9×
  Grande (39–70)   2022           163/471   (34.6%)             45/3686   ( 1.2%)  

## Notas

- A célula de verificação confirma se PSL 2018 = Menos competitivo (8 cadeiras prévias)
- Partidos sem correspondência em bancada_partido_uf.csv recebem bancada_previa=0 → Menos competitivo
- Corrigir no texto: `dm_cat` usa 39–70 para Grande (não 53–70) e Médio é 16–31 (não X–Y)

Os números são fortes. Há três achados que alteram ou qualificam análises anteriores.

### 1. A medida é validada — e mais do que se esperava

Taxa de eleição 14,7× maior para competitivos em 2018, 20,9× em 2022. Isso é suficiente para incluir no texto como validação da medida ex-ante: o partido que aposta em quem tem histórico está apostando em quem de fato vence.

### 2. O achado que complica a narrativa de atenuação

|                                 | 2018  | 2022      |
| ------------------------------- | ----- | --------- |
| % eleitos que eram competitivos | 65,9% | **76,2%** |
| Razão de taxas                  | 14,7× | **20,9×** |

Em 2022, uma proporção *maior* dos eleitos era ex-ante identificável, e a vantagem relativa dos competitivos *aumentou*. Isso parece contradizer a atenuação documentada nos modelos de recursos.

**A reconciliação:** os dois fenômenos medem coisas distintas. Os modelos (Cox e logit) capturam concentração de recursos — que de fato atenuou. Mas a *previsibilidade dos resultados* aumentou porque o fim das coligações eliminou os vencedores por arrastamento: candidatos não-competitivos que antes se elegiam carregados pelo QE coletivo. A taxa de eleição de não-competitivos caiu de 2,6% para 1,5% — esse é o efeito institucional. Resultados mais previsíveis em 2022 não significam mais coordenação; significam menos "ruído eleitoral" de coligação.

### 3. Implicação direta para o bancada solo

Em 2018, 175 eleitos (34,1%) eram "surpresa" — não tinham credenciais ex-ante. Em 2022, apenas 122 (23,8%). Isso significa que a bancada solo em 2022 é ainda mais concentrada entre candidatos com histórico do que em 2018 — o que fortalece o argumento de que a bancada solo em 2022 é um fenômeno mais "limpo" de coordenação do que em 2018.

### O que entra no texto do Cap. 3

O número mais útil para o texto é este: em 2022, **76% dos eleitos já eram reconhecíveis ex-ante** pelo critério de histórico eleitoral. Isso motiva diretamente a discussão do NECr e do M+1 — o partido tinha informação suficiente para coordenar.

### Análise atualizada — tabela com tipo de partido ex-ante

**1. A imagem global não muda: candidatos competitivos são minoria**

11,6% em 2018 e 13,3% em 2022 — mesmos totais, pois o flag individual não muda, apenas a classificação dos partidos. O problema de coordenação existe: 9 em cada 10 candidatos não têm credenciais.

**2. O gap entre tipos de partido permanece, mas fica mais nítido**

|      | Competitivo (média) | Menos comp. (média) | Razão |
| ---- | ------------------- | ------------------- | ----- |
| 2018 | 26,6%               | 7,4%                | 3,6×  |
| 2022 | 23,6%               | 6,9%                | 3,4×  |

Com PSL fora da categoria Competitivo, o denominador de 2018 encolhe e o percentual médio sobe — a categoria fica mais homogênea. O gap permanece ao redor de 3×.

**3. O gradiente de magnitude é limpo e consistente para partidos competitivos**

Competitivo 2018: 30,7% → 28,1% → 21,0% (-9,7pp de pequeno a grande)  
Competitivo 2022: 29,4% → 22,0% → 19,4% (-10pp)

O padrão se repete identicamente nos dois anos: listas maiores diluem os competitivos, pois o preenchimento cresce mais rápido que o núcleo com histórico.

**4. A mudança entre anos: atenuação consistente nos partidos competitivos**

| Magnitude | Δ Competitivo | Δ Menos comp. |
| --------- | ------------- | ------------- |
| Pequeno   | -1,3pp        | -2,8pp        |
| Médio     | **-6,1pp**    | -0,4pp        |
| Grande    | -1,6pp        | +1,6pp        |

A maior queda é no Médio Competitivo (-6,1pp). Isso reflete provavelmente a entrada de PSDB e outros partidos com bancada prévia ≥20 mas poucos incumbentes em 2022 — partidos com estrutura nacional mas com bancada em colapso, que expandiram listas sem expandir o núcleo competitivo.

**5. A assimetria no crescimento de candidatos é o achado mais robusto**

Competitivo: 2.032 → 3.918 candidatos totais (+93%)  
Menos competitivo: 5.598 → 5.757 (+3%)

Partidos competitivos quase dobraram suas listas. Mas o percentual de candidatos competitivos caiu (26,6% → 23,6%). Isso significa que esses partidos expandiram absolvendo candidatos sem histórico — provavelmente incumbentes de partidos extintos pelas fusões somados a candidatos de preenchimento novos. A coordenação se torna mais difícil com mais candidatos por lista, consistente com a atenuação documentada nos modelos.

**6. Consistência com o argumento central**

Sob a classificação ex-ante, a atenuação de 2018→2022 aparece em todas as células de Competitivo — não há segmento onde o percentual sobe. Isso é coerente com o que Cox e logit fracionário mostram: a pressão sobre a concentração cai entre os dois ciclos. A tabela descritiva agora conta a mesma história que os modelos.